In [1]:
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import PyPDFLoader, JSONLoader, CSVLoader, WebBaseLoader, ImageCaptionLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter, TextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from glob import glob

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [ ]:
pdfloader = ImageCaptionLoader(images=['../image/cat.jpg','../image/dog.jpg'],
                               blip_model='Salesforce/blip-image-captioning-large')
document = pdfloader.load()
print( document)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

c:\Users\Dell5371\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Dell5371\.cache\huggingface\hub\models--Salesforce--blip-image-captioning-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

HFValidationError: Repo id must use alphanumeric chars, '-', '_' or '.'. The name cannot start or end with '-' or '.' and the maximum length is 96: 'Salesforce/blip=image=captioning-large'.

In [4]:
embedding = OpenAIEmbeddings( model='text-embedding-ada-002')
vdb = FAISS.from_documents( document, embedding)

In [5]:
# 6. 프롬프트 템플릿 정의
template = """질문에 대해 아래의 제공된 문맥(context)만을 사용하여 답변하세요. 
답을 모른다면 모른다고 말하고 답변을 지어내지 마세요. 
최대한 세 문장 내로 간결하게 답변하세요.

Question: {question} 
Context: {context} 
Answer:
"""
prompt = ChatPromptTemplate.from_template(template)

llm = ChatOpenAI( model='gpt-4o-mini', temperature=0)
retriever = vdb.as_retriever()  # vdb.as_retriever(k=4)

def format_docs(docs):
    return "\n\n".join( doc.page_content for doc in docs )

qa = ( 
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt 
        | llm 
        | StrOutputParser() 
    )
query = "엔지니어 이름은?"
result = qa.invoke( query)
print( result )

엔지니어 이름은 Alice입니다.
